In [0]:
# ===========================================
# Notebook Name:
# 01_ingest_event_results
#
# Purpose:
# Fetch tournament result API responses for
# multiple tournaments selected by the Ops
# fetch-target table.
#
# Input:
# pokemon.ops.result_fetch_targets
#
# Outputs:
# pokemon.bronze.event_result_raw
# pokemon.bronze.scrape_log (source_type='event_result')
#
# Bronze grain:
# One row per tournament_id and response
# version (response_hash).
#
# Idempotency:
# tournament_id + response_hash
# ===========================================

In [0]:
from datetime import datetime, timezone
import hashlib
import json
import time
import uuid

import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

In [0]:
RESULT_FETCH_TARGETS_TABLE = (
    "pokemon.ops.result_fetch_targets"
)

EVENT_RESULT_RAW_TABLE = (
    "pokemon.bronze.event_result_raw"
)

SCRAPE_LOG_TABLE = (
    "pokemon.bronze.scrape_log"
)

RESULT_API_URL_TEMPLATE = (
    "https://players.pokemon-card.com/"
    "event_result/{tournament_id}"
)

RESULT_REFERER_TEMPLATE = (
    "https://players.pokemon-card.com/"
    "event/result/detail/{tournament_id}"
)

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_INTERVAL_SECONDS = 1.0

MAX_FETCH_TARGETS = 500

FAIL_NOTEBOOK_IF_ANY_ERROR = True

print("Input :", RESULT_FETCH_TARGETS_TABLE)
print("Output:", EVENT_RESULT_RAW_TABLE)
print("Log   :", SCRAPE_LOG_TABLE)

In [0]:
targets_df = (
    spark.table(RESULT_FETCH_TARGETS_TABLE)
    .select(
        "tournament_id",
        "fetch_reason",
        "priority",
        "event_date",
    )
    .filter(
        F.col("tournament_id").isNotNull()
    )
    .dropDuplicates(
        ["tournament_id"]
    )
    .orderBy(
        F.col("priority").asc_nulls_last(),
        F.col("event_date").desc_nulls_last(),
        F.col("tournament_id").desc(),
    )
    .limit(
        MAX_FETCH_TARGETS
    )
)

target_count = targets_df.count()

print(
    "Tournament result fetch targets:",
    target_count,
)

display(targets_df)

In [0]:
retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=1.0,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=[
        "GET",
    ],
)

http_adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session = requests.Session()

session.mount(
    "https://",
    http_adapter,
)

session.mount(
    "http://",
    http_adapter,
)

REQUEST_HEADERS = {
    "Accept": (
        "application/json, "
        "text/plain, */*"
    ),
    "Accept-Language": (
        "ja,en-US;q=0.9,en;q=0.8"
    ),
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/150.0.0.0 "
        "Safari/537.36"
    ),
    "X-Accept-Version": "v1",
}

In [0]:
def serialize_payload(
    payload_object,
) -> str:
    return json.dumps(
        payload_object,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def calculate_sha256(
    value: str,
) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

In [0]:
RESULT_LIST_KEYS = [
    "result",
    "results",
    "event_result",
    "event_results",
    "ranking",
    "rankings",
    "players",
]


def find_result_list(
    payload_object,
):
    if not isinstance(
        payload_object,
        dict,
    ):
        return []

    for key in RESULT_LIST_KEYS:
        value = payload_object.get(key)

        if isinstance(value, list):
            return value

    data = payload_object.get("data")

    if isinstance(data, dict):
        for key in RESULT_LIST_KEYS:
            value = data.get(key)

            if isinstance(value, list):
                return value

    return []

In [0]:
def build_result_api_url(
    tournament_id: str,
) -> str:
    return RESULT_API_URL_TEMPLATE.format(
        tournament_id=tournament_id
    )


def fetch_event_result(
    tournament_id: str,
):
    request_url = build_result_api_url(
        tournament_id
    )

    headers = dict(
        REQUEST_HEADERS
    )

    headers["Referer"] = (
        RESULT_REFERER_TEMPLATE.format(
            tournament_id=tournament_id
        )
    )

    started_at = time.perf_counter()

    response = session.get(
        request_url,
        headers=headers,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )

    elapsed_ms = int(
        (
            time.perf_counter()
            - started_at
        ) * 1000
    )

    response.raise_for_status()

    content_type = (
        response.headers
        .get(
            "Content-Type",
            "",
        )
        .lower()
    )

    if "application/json" not in content_type:
        raise ValueError(
            "Tournament result API did not "
            "return JSON. "
            f"tournament_id={tournament_id}, "
            f"status={response.status_code}, "
            f"content_type={content_type}, "
            f"body={response.text[:500]}"
        )

    try:
        payload_object = response.json()

    except ValueError as error:
        raise ValueError(
            "Tournament result response "
            "could not be parsed as JSON. "
            f"tournament_id={tournament_id}, "
            f"body={response.text[:500]}"
        ) from error

    api_code = (
        payload_object.get("code")
        if isinstance(
            payload_object,
            dict,
        )
        else None
    )

    if (
        api_code is not None
        and int(api_code) != 200
    ):
        raise ValueError(
            "Tournament result API returned "
            "a non-success code. "
            f"tournament_id={tournament_id}, "
            f"api_code={api_code}"
        )

    result_items = find_result_list(
        payload_object
    )

    payload = serialize_payload(
        payload_object
    )

    response_hash = calculate_sha256(
        payload
    )

    return {
        "request_url": response.url,
        "response_status": (
            response.status_code
        ),
        "api_code": (
            int(api_code)
            if api_code is not None
            else None
        ),
        "payload": payload,
        "response_hash": response_hash,
        "result_count": len(
            result_items
        ),
        "elapsed_ms": elapsed_ms,
    }

In [0]:
existing_result_versions = {
    (
        row["tournament_id"],
        row["response_hash"],
    )
    for row in (
        spark.table(
            EVENT_RESULT_RAW_TABLE
        )
        .select(
            "tournament_id",
            "response_hash",
        )
        .distinct()
        .collect()
    )
}

print(
    "Existing result response versions:",
    len(existing_result_versions),
)

In [0]:
raw_schema = StructType([
    StructField("tournament_id", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("http_status", IntegerType(), True),
    StructField("payload", StringType(), False),
    StructField("payload_format", StringType(), False),
    StructField("response_hash", StringType(), False),
    StructField("scraped_at", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
])

scrape_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_id", StringType(), True),
    StructField("request_url", StringType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("status", StringType(), False),
    StructField("elapsed_ms", LongType(), True),
    StructField("records_found", IntegerType(), True),
    StructField("error_type", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("scraped_at", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
])

In [0]:
target_rows = targets_df.collect()

raw_rows = []
log_rows = []

success_count = 0
success_empty_count = 0
skipped_count = 0
error_count = 0

for index, target in enumerate(
    target_rows,
    start=1,
):
    tournament_id = target["tournament_id"]
    fetch_reason = target["fetch_reason"] or "unknown"

    run_id = str(uuid.uuid4())
    fetched_at = datetime.now(timezone.utc)

    print(
        f"[{index}/{target_count}] "
        f"tournament_id={tournament_id}, "
        f"reason={fetch_reason}"
    )

    try:
        result = fetch_event_result(
            tournament_id
        )

        result_version = (
            tournament_id,
            result["response_hash"],
        )

        is_new_response = (
            result_version
            not in existing_result_versions
        )

        if result["result_count"] == 0:
            request_status = "success_empty"
            success_empty_count += 1

        elif is_new_response:
            request_status = "success"
            success_count += 1

        else:
            request_status = "skipped_unchanged"
            skipped_count += 1

        if is_new_response:
            raw_rows.append({
                "tournament_id": tournament_id,
                "source_url": result["request_url"],
                "http_status": result["response_status"],
                "payload": result["payload"],
                "payload_format": "json",
                "response_hash": result["response_hash"],
                "scraped_at": fetched_at,
                "ingestion_date": fetched_at.date(),
            })

            existing_result_versions.add(
                result_version
            )

        log_rows.append({
            "run_id": run_id,
            "source_type": "event_result",
            "source_id": tournament_id,
            "request_url": result["request_url"],
            "http_status": result["response_status"],
            "status": request_status,
            "elapsed_ms": result["elapsed_ms"],
            "records_found": result["result_count"],
            "error_type": None,
            "error_message": None,
            "scraped_at": fetched_at,
            "ingestion_date": fetched_at.date(),
        })

        print(
            f"  status={request_status}, "
            f"result_count={result['result_count']}, "
            f"new_response={is_new_response}"
        )

    except Exception as error:
        error_count += 1

        error_message = (
            f"{type(error).__name__}: "
            f"{str(error)}"
        )[:4000]

        log_rows.append({
            "run_id": run_id,
            "source_type": "event_result",
            "source_id": tournament_id,
            "request_url": build_result_api_url(
                tournament_id
            ),
            "http_status": None,
            "status": "error",
            "elapsed_ms": None,
            "records_found": 0,
            "error_type": type(error).__name__,
            "error_message": error_message,
            "scraped_at": fetched_at,
            "ingestion_date": fetched_at.date(),
        })

        print(
            f"  status=error, error={error_message}"
        )

    if index < target_count:
        time.sleep(
            REQUEST_INTERVAL_SECONDS
        )

print("Success count:", success_count)
print("Success empty count:", success_empty_count)
print("Skipped count:", skipped_count)
print("Error count:", error_count)

In [0]:
if raw_rows:
    event_result_raw_df = (
        spark.createDataFrame(
            raw_rows,
            schema=raw_schema,
        )
    )

    (
        event_result_raw_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            EVENT_RESULT_RAW_TABLE
        )
    )

    print(
        "Inserted event_result_raw rows:",
        len(raw_rows),
    )

else:
    print(
        "No new event_result_raw rows to insert."
    )

In [0]:
if log_rows:
    scrape_log_df = (
        spark.createDataFrame(
            log_rows,
            schema=scrape_log_schema,
        )
    )

    (
        scrape_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            SCRAPE_LOG_TABLE
        )
    )

    print(
        "Inserted scrape_log rows:",
        len(log_rows),
    )

else:
    print(
        "No scrape_log rows to insert."
    )

In [0]:
run_ids_this_run = [
    row["run_id"] for row in log_rows
]

tournament_ids_fetched_this_run = [
    row["tournament_id"] for row in raw_rows
]

if tournament_ids_fetched_this_run:
    display(
        spark.table(
            EVENT_RESULT_RAW_TABLE
        )
        .filter(
            F.col("tournament_id").isin(
                tournament_ids_fetched_this_run
            )
        )
        .orderBy(
            F.col("scraped_at").desc()
        )
    )
else:
    print("No new event_result_raw rows this run.")

In [0]:
display(
    spark.table(
        SCRAPE_LOG_TABLE
    )
    .filter(
        F.col("run_id").isin(
            list(run_ids_this_run)
        )
    )
    .orderBy(
        F.col("scraped_at").desc()
    )
)

In [0]:
duplicate_versions_df = (
    spark.table(
        EVENT_RESULT_RAW_TABLE
    )
    .groupBy(
        "tournament_id",
        "response_hash",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_count = (
    duplicate_versions_df.count()
)

if duplicate_count > 0:
    display(
        duplicate_versions_df
    )

    raise ValueError(
        f"{duplicate_count} duplicate "
        "event result response versions "
        "detected"
    )

print(
    "Validation passed: "
    "no duplicate tournament response "
    "versions"
)

In [0]:
if (
    FAIL_NOTEBOOK_IF_ANY_ERROR
    and error_count > 0
):
    raise RuntimeError(
        "Some tournament result requests "
        "failed. "
        f"error_count={error_count}"
    )